In [ ]:
!nvidia-smi
!pip install unsloth trl transformers datasets accelerate peft requests -q
print("All installed!")

Sat Apr 25 14:27:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             31W /   70W |    6807MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "v_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("Model loaded!")

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded!


In [ ]:
!git clone https://github.com/letsjoyn/meta-scalar-hack.git
import sys
sys.path.insert(0, "/content/meta-scalar-hack")
print("Repo ready!")

fatal: destination path 'meta-scalar-hack' already exists and is not an empty directory.
Repo ready!


In [ ]:
SYSTEM_PROMPT = """\
You are an Emergency Operations Center dispatcher. You process disaster incidents one action at a time.

STRICT RULES:
1. Output ONLY a single JSON object. No explanation, no markdown, no extra text.
2. Use ONLY these action_type values: classify, set_priority, draft_reply, submit_ticket, next_ticket, finish_episode
3. Use ONLY these predicted_team values: rescue, medical, utilities, shelter, logistics, general
4. Use ONLY these predicted_priority values: low, medium, high, urgent
5. Always copy the ticket_id EXACTLY from the observation.

WORKFLOW per ticket (follow this order):
  Step 1 → {"action_type": "classify", "ticket_id": "<id>", "predicted_team": "<team>"}
  Step 2 → {"action_type": "set_priority", "ticket_id": "<id>", "predicted_priority": "<priority>"}
  Step 3 → {"action_type": "draft_reply", "ticket_id": "<id>", "reply_text": "<handoff note with dispatch, ETA, resources>"}
  Step 4 → {"action_type": "submit_ticket", "ticket_id": "<id>"}
  Step 5 → {"action_type": "next_ticket"}

After all tickets: {"action_type": "finish_episode"}

TEAM ROUTING GUIDE:
- rescue: trapped people, floods, evacuation, stranded, dam overflow, chemical plume
- medical: injuries, hospital, triage, ambulance, pileup, patients
- utilities: power, transformer, gas line, grid, generator, cold-chain, communication tower
- shelter: evacuees, shelter capacity, water shortage
- logistics: bus, fuel, transport, route coordination
"""
print("System prompt ready!")

System prompt ready!


In [ ]:
import json
import re

VALID_ACTION_TYPES = {"classify", "set_priority", "draft_reply", "submit_ticket", "next_ticket", "finish_episode"}
VALID_TEAMS       = {"rescue", "medical", "utilities", "shelter", "logistics", "general"}
VALID_PRIORITIES  = {"low", "medium", "high", "urgent"}

def parse_action_safe(text):
    text = text.strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
    if match:
        try:
            obj = json.loads(match.group())
            if isinstance(obj, dict):
                return obj
        except Exception:
            pass
    return None

def reward_fn(completions, prompts=None, gold_team=None, gold_priority=None, ticket_id=None, **kwargs):
    rewards = []
    for i, completion in enumerate(completions):
        if isinstance(completion, list):
            text = completion[0].get("content", "") if completion else ""
        else:
            text = str(completion)

        g_team = gold_team[i] if gold_team else None
        t_id   = ticket_id[i] if ticket_id else None
        action = parse_action_safe(text)

        if action is None:
            rewards.append(0.0)
            continue

        score = 0.0
        score += 0.20  # valid JSON

        at = action.get("action_type", "")
        if at == "classify":
            score += 0.10
        elif at not in VALID_ACTION_TYPES:
            score -= 0.10

        if t_id and str(action.get("ticket_id", "")).strip() == str(t_id).strip():
            score += 0.10

        predicted_team = action.get("predicted_team", "")
        if predicted_team == g_team:
            score += 0.40
        elif predicted_team in VALID_TEAMS:
            score += 0.05

        non_ascii = sum(1 for c in text if ord(c) > 127)
        if non_ascii / max(len(text), 1) < 0.1:
            score += 0.20

        rewards.append(max(0.0, min(1.0, score)))

    return rewards

print("Reward function ready!")

Reward function ready!


In [ ]:
from tasks import TASKS
from datasets import Dataset

def build_training_dataset():
    examples = []
    for difficulty, task_spec in TASKS.items():
        for ticket in task_spec.tickets:
            user_msg = f"""CURRENT INCIDENT — {difficulty.upper()} tier

Ticket ID: {ticket.ticket_id}
Message: {ticket.customer_message}
Region criticality: {ticket.customer_tier}

What is your first action? Output JSON only."""

            examples.append({
                "prompt": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
                "ticket_id": ticket.ticket_id,
                "gold_team": ticket.gold_team,
                "gold_priority": ticket.gold_priority,
            })
    return examples

raw_dataset = build_training_dataset()
hf_dataset = Dataset.from_list(raw_dataset)

def apply_template(example):
    example["prompt"] = tokenizer.apply_chat_template(
        example["prompt"],
        tokenize=False,
        add_generation_prompt=True,
    )
    return example

hf_dataset = hf_dataset.map(apply_template)
print(f"Dataset ready: {len(hf_dataset)} examples")

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Dataset ready: 15 examples


In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="disaster_grpo_output",
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=80,      # ← changed from max_new_tokens
    max_prompt_length=1024,
    num_train_epochs=3,
    save_strategy="epoch",
    logging_steps=1,
    report_to="none",
    temperature=0.7,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=hf_dataset,
)

print("Starting training!")
trainer.train()

Starting training!


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 15 | Num Epochs = 3 | Total steps = 45
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 5,046,272 of 7,620,662,784 (0.07% trained)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transform

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_fn / mean,rewards / reward_fn / std
1,-0.000000,0.737500,0.175000,27.000000,27.000000,27.000000,0.000000,27.000000,27.000000,27.000000,0.000008,0.737500,0.175000
2,-0.000000,1.000000,0.000000,26.000000,26.000000,26.000000,0.000000,26.000000,26.000000,26.000000,-0.000000,1.000000,0.000000
3,-0.000000,0.650000,0.000000,27.000000,27.000000,27.000000,0.000000,27.000000,27.000000,27.000000,-0.000000,0.650000,0.000000
4,-0.000000,1.000000,0.000000,26.000000,26.000000,26.000000,0.000000,26.000000,26.000000,26.000000,-0.000000,1.000000,0.000000
5,0.000000,1.000000,0.000000,26.000000,26.000000,26.000000,0.000000,26.000000,26.000000,26.000000,0.000000,1.000000,0.000000
6,-0.000000,1.000000,0.000000,27.000000,27.000000,27.000000,0.000000,27.000000,27.000000,27.000000,-0.000000,1.000000,0.000000
7,0.000000,0.650000,0.000000,26.000000,26.000000,26.000000,0.000000,26.000000,26.000000,26.000000,0.000000,0.650000,0.000000
8,-0.000000,1.000000,0.000000,26.000000,26.000000,26.000000,0.000000,26.000000,26.000000,26.000000,-0.000000,1.000000,0.000000
9,0.000000,1.000000,0.000000,26.000000,26.000000,26.000000,0.000000,26.000000,26.000000,26.000000,0.000000,1.000000,0.000000
10,-0.000000,1.000000,0.000000,27.000000,27.000000,27.000000,0.000000,27.000000,27.000000,27.000000,-0.000000,1.000000,0.000000


Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

TrainOutput(global_step=45, training_loss=1.558392590684382e-08, metrics={'train_runtime': 597.6435, 'train_samples_per_second': 0.075, 'train_steps_per_second': 0.075, 'total_flos': 0.0, 'train_loss': 1.558392590684382e-08})

In [ ]:
import os
from huggingface_hub import login

login(token=os.environ.get("HF_TOKEN"))

# Save LoRA adapters only (no merging, no OOM)
model.save_pretrained("disaster_lora_adapters")
tokenizer.save_pretrained("disaster_lora_adapters")

# Push adapters to hub
model.push_to_hub(
    "joynnayvedya/disaster-response-trained",
    token=os.environ.get("HF_TOKEN"),
)
tokenizer.push_to_hub(
    "joynnayvedya/disaster-response-trained",
    token=os.environ.get("HF_TOKEN"),
)

print("Done! LoRA adapters pushed to hub.")

Unsloth: Restored added_tokens_decoder metadata in disaster_lora_adapters/tokenizer_config.json.


README.md:   0%|          | 0.00/586 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   3%|2         |  574kB / 20.2MB            

...

Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpw7kwz72e/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpw7kwz72e/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Done! LoRA adapters pushed to hub.
